[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/08-rebuild/08_data_vis/08_4_Comparing_Groups.ipynb)

# 08.4: Comparing Groups

Notebook 08.3 showed one numeric column with a histogram. Many questions ask how a column differs across groups, such as whether one species is heavier than another. A bar of means gives one number per group, and notebook 08.2 showed how much that can hide. This notebook introduces the **box plot**, which summarizes each group in a form that lines up well for comparison. It also shows what a box plot leaves out, and what to do about it.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## Reading a box plot

A box plot draws five numbers from `.describe()` and marks any unusual values:

- The **box** runs from the 25th percentile to the 75th percentile, so it holds the middle half of the values.
- The **line inside the box** is the median.
- The **whiskers** are the lines extending from the box. They reach the furthest values that are within 1.5 box lengths of the box.
- **Points** beyond the whiskers are drawn one by one as possible outliers.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_penguins.csv")
print(penguins.groupby("species")["body_mass_g"].describe())
ax = sns.boxplot(data=penguins, x="species", y="body_mass_g")
ax.set_title("Body mass by species")
ax.set_xlabel("Species")
ax.set_ylabel("Body mass (grams)")
plt.show()

Match the table to the picture. For Gentoo penguins, the `25%`, `50%` and `75%` values of 4700, 5000 and 5500 are the bottom of the box, the line and the top of the box. The Gentoo box sits entirely above the other two, so nearly all Gentoo penguins are heavier than a typical penguin of either other species. The Adelie and Chinstrap boxes overlap almost completely, with both medians at 3700 grams. The two points beyond the Chinstrap whiskers, at 2700 and 4800 grams, are outliers by the 1.5 box length rule. An outlier is a value to look at, and it does not have to be an error. A penguin can simply be unusually large.

## What a bar of means hid

In notebook 08.2, a bar chart suggested that Broadway specials earn a little more per week than plays. Here is the same comparison with a box plot.

In [ ]:
broadway = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
ax = sns.boxplot(data=broadway, x="Type", y="Gross")
ax.set_title("Weekly gross by type of show")
ax.set_xlabel("Type of show")
ax.set_ylabel("Weekly gross (dollars)")
plt.show()
print(broadway.groupby("Type")["Gross"].describe()[["count", "25%", "50%", "75%"]])

The `1e6` at the top of the axis means that the axis numbers are in millions, so 1.0 is one million dollars. Now you can see that the two groups differ in a way the bars could not show. The median for specials, about 240084, is a little below the median for plays, about 262396, and the box for specials is much taller. It runs from about 65546 to about 483034, while the box for plays runs from about 176337 to about 399726. So specials are much more varied, with many small weeks and some very large ones. That is a better answer to the question than "specials earn a little more."

## What a box plot hides

A box plot gets its compact form by leaving out the shape. Here is the Old Faithful column from notebook 08.3, which has two clear peaks, drawn as a box plot.

In [ ]:
geyser = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_geyser.csv")
ax = sns.boxplot(data=geyser, x="duration")
ax.set_title("Duration of Old Faithful eruptions")
ax.set_xlabel("Duration (minutes)")
plt.show()

ax = sns.histplot(data=geyser, x="duration", binwidth=0.25)
ax.set_title("The same column as a histogram")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of eruptions")
plt.show()

The box runs from about 2.16 to about 4.45 minutes with a median of 4.0, and nothing in it suggests two groups. In fact, the box suggests that the middle of its range is where many values sit, when that stretch is nearly empty. So a box plot alone cannot tell you whether a column has one peak or two. Use a histogram first to learn the shape of a column, and then use box plots to compare groups once you know that a summary is reasonable.

In this table, the `kind` column separates the eruptions into short and long, so the two peaks can be shown as two groups. Splitting a column into the groups that explain its peaks is often the best fix.

In [ ]:
ax = sns.boxplot(data=geyser, x="duration", y="kind")
ax.set_title("Duration by kind of eruption")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Kind of eruption")
plt.show()

The two boxes do not overlap at all, and each one now describes a real group: short eruptions with a median near 2 minutes and long eruptions with a median a little above 4.

## A box does not show how many rows are behind it

Every box looks equally solid, whether it summarizes 1000 rows or 3. Here are the median earnings of majors for each category. The categories have no natural order, so we sort them by their medians using `order=`, and we put the category on the y axis so that the labels can be read.

In [ ]:
majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
pay_order = majors.groupby("Major_category")["Median"].median().sort_values(ascending=False).index
ax = sns.boxplot(data=majors, x="Median", y="Major_category", order=pay_order)
ax.set_title("Median earnings of majors, by category")
ax.set_xlabel("Median earnings (dollars)")
ax.set_ylabel("")
plt.show()
majors["Major_category"].value_counts().tail(4)

Engineering has the highest median at 57000 dollars, and its long reach to the right comes from the outlier at 110000, which is Petroleum Engineering. Now look at Interdisciplinary, which is just a single line at 35000. The count shows that it has only 1 major, so there is no spread to draw at all. Communications & Journalism has only 4 majors, and Law & Public Policy only 5, so their boxes are much less reliable than the one for Engineering, which has 29. The box plot gives no sign of this difference.

When groups are small, you can show the rows themselves. A **strip plot** draws one point per row, with some random sideways shift, called **jitter**, so that points with the same value do not land on top of one another. Drawing it over a box plot keeps the summary and shows the rows. `showfliers=False` turns off the outlier points on the box plot, since the strip plot now draws them.

In [ ]:
ax = sns.boxplot(data=majors, x="Median", y="Major_category", order=pay_order, showfliers=False, color="lightgray")
sns.stripplot(data=majors, x="Median", y="Major_category", order=pay_order, size=3)
ax.set_title("Median earnings of majors, with one point per major")
ax.set_xlabel("Median earnings (dollars)")
ax.set_ylabel("")
plt.show()

Now you can see how little is behind some of the boxes, such as the single point for Interdisciplinary and the 4 points for Communications & Journalism, and how much is behind Engineering. A strip plot works well with up to about a few dozen points per group. Beyond that the points run together, and with 22551 musicals you would see only a solid band.

## Histograms for groups

Sometimes you want the shape of each group and not only a summary. The first idea is to color a single histogram by group with `hue=`.

In [ ]:
ax = sns.histplot(data=penguins, x="flipper_length_mm", hue="species")
ax.set_title("Flipper length, one chart for three species")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Number of penguins")
plt.show()

The overlapping colors mix together, so it is hard to tell which group a bar belongs to where they cross, between about 185 and 215 mm. It works better to give each group its own small chart. `sns.displot()` draws histograms, and `col=` makes one chart per value of a column, using the same axes in all of them so that they can be compared directly. Like other functions that draw several charts, it returns a different kind of object, so its labels are set with `grid.set_axis_labels()`, and `grid.set_titles()` sets the title of each small chart.

In [ ]:
grid = sns.displot(data=penguins, x="flipper_length_mm", col="species", binwidth=5, height=3)
grid.set_axis_labels("Flipper length (mm)", "Number of penguins")
grid.set_titles("{col_name}")
plt.show()

The shapes are now easy to compare. Each species has one peak: Adelie penguins near 190 mm, Chinstrap penguins near 195 mm and Gentoo penguins near 215 mm. This also explains the dip in the middle of the combined data, since Adelie and Chinstrap penguins make up the lower peak and Gentoo penguins make up the higher one. Because the three charts count rows, the shorter bars for Chinstrap penguins partly reflect that there are only 68 of them.

## Choosing among them

| Situation | Chart |
|---|---|
| Many groups, and you want to compare centers and spreads | Box plots, sorted if the groups have no natural order |
| A few groups, and the shape of each one matters | One histogram per group |
| Small groups, or you need to show how many rows are behind each group | Strip plot, with or without a box plot |
| The question is only about a single summary, and you have checked the shapes | Bar chart of that summary |

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_penguins.csv")
majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
geyser = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_geyser.csv")
category_order = majors.groupby("Major_category")["ShareWomen"].median().sort_values(ascending=False).index.to_list()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
Make a box plot of `bill_length_mm` for each `species`, using `penguins`, and store it in `bill_box`. Give it a title and label both axes.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check_chart(1, "bill_box", lambda c: {t.get_text() for t in c.get_xticklabels() + c.get_yticklabels()} >= {"Adelie", "Chinstrap", "Gentoo"} and max(c.get_xlim() + c.get_ylim()) < 70, "Use sns.boxplot() with species on one axis and bill_length_mm on the other.")

### Problem 2
Store the median `bill_length_mm` of Adelie penguins in a variable named `adelie_median`. You can check your answer against the line in the box from Problem 1.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "adelie_median", penguins[penguins["species"] == "Adelie"]["bill_length_mm"].median(), "Filter to Adelie with a mask, or group by species, and take the median.")

### Problem 3
`category_order` lists the categories of majors from the highest median `ShareWomen` to the lowest. Make a horizontal box plot of `ShareWomen` for each `Major_category`, in that order, and store it in `share_box`. Give it a title and an x axis label.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check_chart(3, "share_box", lambda c: [t.get_text() for t in c.get_yticklabels()] == category_order, "Put ShareWomen on x and Major_category on y, and add order=category_order.", y_label=False)

### Problem 4
Store the name of the `Major_category` with the fewest rows in a variable named `smallest_group`.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "smallest_group", majors["Major_category"].value_counts().index[-1], ".value_counts() sorts from most common to least, so look at the last label in its index.")

### Problem 5
Draw a strip plot of `Median` for each `Major_category` over a box plot of the same columns, and store the chart in `pay_strip`. Use `showfliers=False` on the box plot. Give it a title and an x axis label.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check_chart(5, "pay_strip", lambda c: sum(len(p.get_offsets()) for p in c.collections) >= 172, "Call sns.boxplot() with showfliers=False, and then sns.stripplot() with the same x and y.", y_label=False)

### Problem 6
You want to find out whether the `waiting` column of `geyser` has one peak or two. Which chart do you draw first? Store your answer in a variable named `peaks_chart`, as one of `"box"` or `"histogram"`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "peaks_chart", "histogram", "A box plot leaves out the shape of a column.")

### Problem 7
Make that chart, using `geyser` and a `binwidth` of 5, and store it in `waiting_hist`. Give it a title and label both axes.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check_chart(7, "waiting_hist", lambda c: 4 < c.patches[0].get_width() < 6 and sum(p.get_height() for p in c.patches) == 272, "Use sns.histplot() with x=\"waiting\" and binwidth=5.")

## What's next

You can now compare a numeric column across groups and know what your chart is leaving out. So far, each chart has looked at one numeric column at a time. Notebook 08.5 looks at two numeric columns together: whether they are related, and how a chart or a single number can mislead you about that relationship.